In [8]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import ast

In [10]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [12]:
movies = movies.merge(credits, on="title")

In [14]:
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]

In [16]:
movies.dropna(inplace=True)

In [18]:
def convert(text):

    L = []

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [20]:
movies['genres'] = movies['genres'].apply(convert)

In [22]:
movies['keywords'] = movies['keywords'].apply(convert)

In [24]:
def convert_cast(text):

    L = []

    counter = 0

    for i in ast.literal_eval(text):

        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [26]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [27]:
def fetch_director(text):

    L = []

    for i in ast.literal_eval(text):

        if i['job'] == 'Director':

            L.append(i['name'])

    return L

In [30]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [31]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [34]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])

movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])

movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])

movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [36]:
movies['tags'] = movies['overview'] + \
                 movies['genres'] + \
                 movies['keywords'] + \
                 movies['cast'] + \
                 movies['crew']

In [38]:
new_df = movies[['movie_id','title','tags']]

In [40]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())
cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)

C:\Users\Jhanvi\AppData\Local\Temp\ipykernel_10008\501331008.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))
C:\Users\Jhanvi\AppData\Local\Temp\ipykernel_10008\501331008.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [42]:
def recommend(movie):

    movie = movie.lower()

    index = new_df[new_df['title'].str.lower() == movie].index

    if len(index) == 0:
        print("Movie not found!")
        return

    index = index[0]

    distances = list(enumerate(similarity[index]))

    movies_list = sorted(distances,
                         reverse=True,
                         key=lambda x:x[1])[1:6]

    print("\nRecommended Movies:\n")

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [44]:
recommend("Avatar")


Recommended Movies:

Titan A.E.
Small Soldiers
Ender's Game
Aliens vs Predator: Requiem
Independence Day


In [46]:
recommend("Iron Man")


Recommended Movies:

Iron Man 2
Iron Man 3
Avengers: Age of Ultron
Captain America: Civil War
The Avengers


In [48]:
recommend("Frozen")


Recommended Movies:

Spirit: Stallion of the Cimarron
Aladdin
The Book of Life
Curious George
Delgo
